In [116]:
# 1. IMPORT LIBRARIES
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

In [117]:
# 2. DOWNLOAD DATA

# Get NIFTY 50 data
df = yf.download("^NSEI",
                 start="2019-06-06",
                 end="2026-01-01",
                 auto_adjust=False,
                 multi_level_index=False)

# Keep only adjusted close
df = df[['Adj Close']]

# 3. FEATURE ENGINEERING

# Log returns → more stable for modeling
df['returns'] = np.log(df['Adj Close'] / df['Adj Close'].shift(1))

# Lagged return (yesterday's return)
df['lag1'] = df['returns'].shift(1)

# Rolling mean → short-term trend (shift to avoid leakage)
df['ma_20'] = df['returns'].rolling(20).mean().shift(1)

# Rolling volatility → risk measure (shift to avoid leakage)
df['volatility_20'] = df['returns'].rolling(20).std().shift(1)

# Drop NaNs caused by rolling/shift
df.dropna(inplace=True)

# 4. DEFINE FEATURES & TARGET

X = df[['lag1', 'ma_20', 'volatility_20']]
y = df['returns']

window = 220
coef = []
pred = []
date = []
intercept = []

for i in range(window,len(df)):

    X_train = X.iloc[i-window:i]
    y_train = y.iloc[i-window:i]

    x_test = X.iloc[i:i+1]
    model = LinearRegression().fit(X_train, y_train)
    forecast = model.predict(x_test)

    pred.append(forecast[0])
    coef.append(model.coef_)
    date.append(x_test.index[0])
    intercept.append(model.intercept_)
coef = np.array(coef)
intercept = np.array(intercept)

[*********************100%***********************]  1 of 1 completed


In [118]:
results = pd.DataFrame({"prediction":pred,"coef_1":coef[:,0],"coef_2":coef[:,1],"coef_3":coef[:,2],"intercept":intercept},index=date)
results['actual'] = df.loc[results.index,'returns']
results['residual'] = results['actual'] - results['prediction']
results['prediction_direction'] = np.where(results['prediction']>0,1,-1)
results['actual_direction'] = np.where(results['actual']>0,1,-1)
accuracy = (results['prediction_direction'] == results['actual_direction']).mean()
results['abs_prediction'] = abs(results['prediction'])
results_sorted = results.sort_values(by='abs_prediction', ascending=False)
top_n = int(len(results_sorted) * 0.2)
strong_signals = results_sorted.iloc[:top_n]
strong_accuracy = (strong_signals['prediction_direction'] == strong_signals['actual_direction']).mean()
print(strong_accuracy)


0.4927536231884058


In [119]:
naive_direction = np.where(df.loc[results.index, 'lag1'] > 0, 1, -1)
naive_accuracy = (naive_direction == results['actual_direction']).mean()
print(naive_accuracy)

0.5621387283236994
